# Task 1 — Inconsistencies and duplicates

**Input:** `club_signups.csv` (never edited). **Output:** `club_signups_clean.csv`, one row per student per club.

Every fix below is done in code.

In [1]:
import pandas as pd

raw = pd.read_csv("club_signups.csv", dtype=str)   # read everything as text first, so nothing is guessed
df = raw.copy()                                    # work on a copy; `raw` stays as it was in the file

## Part A — Inconsistencies

### A1. Look first

In [2]:
print("shape:", df.shape)
df.dtypes

shape: (39, 9)


signup_id       str
student_id      str
full_name       str
email           str
faculty         str
city            str
club            str
signed_up_at    str
fee_paid        str
dtype: object

In [3]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False), "\n")

--- faculty ---
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

--- club ---
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Chess Club     3
Football       3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64 

--- city ---
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza  

In [4]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: type, missing values, distinct values and the most common values."""
    rows = {}
    for name, column in frame.items():
        rows[name] = {
            "dtype": str(column.dtype),
            "missing": column.isna().sum(),
            "distinct": column.nunique(),
            "most common": column.value_counts().head(3).to_dict(),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


profile(raw)

,dtype,missing,distinct,most common
signup_id,str,0,36,"{'S015': 2, 'S022': 2, 'S023': 2}"
student_id,str,0,15,"{'61-7344': 6, '55-9245': 5, '61-4844': 4}"
full_name,str,0,24,"{'Laila Mostafa': 4, 'Hana Zaki': 3, 'Ziad Sal..."
email,str,0,18,"{'ali.ibrahim@student.guc.edu.eg': 5, 'mohamed..."
faculty,str,0,13,"{'Pharmacy': 7, 'EMS': 7, 'IET': 5}"
city,str,0,9,"{'Cairo': 13, 'Alexandria': 11, 'Giza': 4}"
club,str,0,13,"{'Robotics': 8, 'Music': 5, 'Chess': 4}"
signed_up_at,str,0,36,"{'2026-09-15 13:39': 2, '2026-09-16 01:39': 2,..."
fee_paid,str,0,9,"{'yes': 7, '1': 7, 'false': 6}"


In [5]:
# The other columns, briefly: names and emails have extra spaces and mixed capitals,
# and signed_up_at mixes two date formats.
df[["full_name", "email", "signed_up_at"]].head(10)

,full_name,email,signed_up_at
0,Hana Zaki,hana.zaki@student.guc.edu.eg,2026-09-15 13:39
1,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,15/09/2026 03:12
2,Mohamed Adel,mohamed.adel@student.guc.edu.eg,2026-09-14 23:32
3,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,2026-09-17 11:15
4,Laila Mostafa,laila.mostafa@student.guc.edu.eg,2026-09-13 22:36
5,ALI IBRAHIM,ali.ibrahim@student.guc.edu.eg,15/09/2026 03:42
6,ahmed adel,AHMED.ADEL@STUDENT.GUC.EDU.EG,2026-09-14 17:03
7,OMAR ADEL,omar.adel@student.guc.edu.eg,2026-09-13 23:52
8,ali ibrahim,ali.ibrahim@student.guc.edu.eg,2026-09-15 23:07
9,ali ibrahim,ali.ibrahim@student.guc.edu.eg,2026-09-16 09:07


In [6]:
print("signup_id unique?", raw["signup_id"].is_unique)
print(f"{len(raw):,} rows, {raw['signup_id'].nunique():,} distinct signup ids")

signup_id unique? False
39 rows, 36 distinct signup ids


### A2. Faculty, club, city

1. **Mechanical normalisation:** strip outer spaces, lower-case, and collapse repeated inner spaces.
2. **Explicit dictionary** for what is left (full names, abbreviations, synonyms such as *Soccer → Football*).

In [7]:
def normalise(s: pd.Series) -> pd.Series:
    """'  Chess  Club ' -> 'chess club'"""
    return s.str.strip().str.lower().str.split().str.join(" ")

FACULTY = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
CLUB = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
CITY = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

for col, mapping in [("faculty", FACULTY), ("club", CLUB), ("city", CITY)]:
    df[col] = normalise(df[col]).map(mapping)

# Only the canonical values may remain
assert set(df["faculty"]) <= {"MET", "IET", "EMS", "Pharmacy", "Management"}
assert set(df["club"])    <= {"Robotics", "Debate", "Music", "Football", "Chess"}
assert set(df["city"])    <= {"Cairo", "Giza", "Alexandria"}

for col in ["faculty", "club", "city"]:
    print(df[col].value_counts(), "\n")

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64 

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64 

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64 



### A3. Names and emails

Names: remove all extra spaces (outer and inner), then Title Case for everyone. Emails: trim and lower-case.

In [8]:
df["full_name"] = df["full_name"].str.split().str.join(" ").str.title()   # '  Mohamed  Adel ' -> 'Mohamed Adel'
df["email"] = df["email"].str.strip().str.lower()

df[["student_id", "full_name", "email"]].drop_duplicates().sort_values("student_id")

,student_id,full_name,email
27,55-2639,Youssef Fathy,youssef.fathy@student.guc.edu.eg
19,55-2971,Salma Ibrahim,salma.ibrahim@student.guc.edu.eg
14,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg
5,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg
26,58-1089,Karim Kamel,karim.kamel@student.guc.edu.eg
12,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg
16,58-8701,Farida Ibrahim,farida.ibrahim@student.guc.edu.eg
17,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg
6,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg
1,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg


### A4. `fee_paid` → boolean

In [9]:
FEE = {"yes": True, "y": True, "true": True, "1": True,
       "no": False, "n": False, "false": False, "0": False}

fee = df["fee_paid"].str.strip().str.lower()
assert set(fee) <= set(FEE), f"unknown fee values: {set(fee) - set(FEE)}"
df["fee_paid"] = fee.map(FEE).astype(bool)

print(df["fee_paid"].dtype)
df["fee_paid"].value_counts()

bool


fee_paid
True     27
False    12
Name: count, dtype: int64

### A5. `signed_up_at` → one datetime column

Find the formats first.

In [10]:
s = df["signed_up_at"].str.strip()
iso = s.str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$")   # 2026-09-15 13:39
slash = s.str.match(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$") # 15/09/2026 03:12

print("YYYY-MM-DD HH:MM :", iso.sum())
print("NN/NN/YYYY HH:MM :", slash.sum())

# Which part of NN/NN is the day? The first part goes above 12, so it can't be a month.
first = s[slash].str[:2].astype(int)
second = s[slash].str[3:5].astype(int)
print("first part ranges", first.min(), "-", first.max(), "| second part ranges", second.min(), "-", second.max())

YYYY-MM-DD HH:MM : 25
NN/NN/YYYY HH:MM : 14
first part ranges 13 - 18 | second part ranges 9 - 9


The first part reaches values above 12 and the second part is always 09 (the sign-up week was in September), so the slash format is **day/month/year**: `%d/%m/%Y`.

In [11]:
parsed_iso = pd.to_datetime(s.where(iso), format="%Y-%m-%d %H:%M")
parsed_slash = pd.to_datetime(s.where(slash), format="%d/%m/%Y %H:%M")
df["signed_up_at"] = parsed_iso.fillna(parsed_slash)

assert df["signed_up_at"].notna().all()
print(df["signed_up_at"].dtype, "| from", df["signed_up_at"].min(), "to", df["signed_up_at"].max())

datetime64[us] | from 2026-09-13 14:36:00 to 2026-09-18 23:36:00


## Part B — Duplicates

### B1. Exact duplicates (the same submission twice)

In [12]:
rows_start = len(df)
exact = df.duplicated()
print("rows at start:", rows_start)
print("exact duplicate rows:", exact.sum())
display(df[df.duplicated(keep=False)].sort_values("signup_id"))

df = df.drop_duplicates()
print("rows after removing exact duplicates:", len(df))

rows at start: 39
exact duplicate rows: 3


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


rows after removing exact duplicates: 36


### B2. The same sign-up submitted again

Same `student_id` + `club`, different `signup_id`. Keep the **latest** submission (it carries the updated fee status).

In [13]:
repeat = df[df.duplicated(["student_id", "club"], keep=False)].sort_values(["student_id", "club", "signed_up_at"])
display(repeat[["signup_id", "student_id", "full_name", "club", "signed_up_at", "fee_paid"]])

before = len(df)
df = (df.sort_values("signed_up_at")
        .drop_duplicates(subset=["student_id", "club"], keep="last")   # last = latest after sorting by time
        .sort_values("signup_id")
        .reset_index(drop=True))
print("rows removed by B2:", before - len(df))
print("rows left:", len(df))

,signup_id,student_id,full_name,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,Football,2026-09-18 23:36:00,True


rows removed by B2: 4
rows left: 32


### B3. Check

In [14]:
assert not df.duplicated(["student_id", "club"]).any(), "student_id + club is not unique"
print("student_id + club is unique ✓")

student_id + club is unique ✓


### Why the order matters

Repeat the duplicate steps on the **raw** data, before the club spellings are fixed:

In [15]:
raw_dedup = raw.drop_duplicates()
found_early = len(raw_dedup) - len(raw_dedup.drop_duplicates(["student_id", "club"]))
print("exact duplicates on raw data:", len(raw) - len(raw_dedup))
print("same-sign-up duplicates found BEFORE fixing clubs:", found_early, "(vs", before - len(df), "after fixing)")

exact duplicates on raw data: 3
same-sign-up duplicates found BEFORE fixing clubs: 2 (vs 4 after fixing)


### Same name, different students

In [16]:
same_name = df.groupby("full_name")["student_id"].nunique()
names = same_name[same_name > 1].index
df[df["full_name"].isin(names)][["signup_id", "student_id", "full_name", "faculty", "city", "club"]]

,signup_id,student_id,full_name,faculty,city,club
1,S002,55-2992,Mohamed Adel,IET,Alexandria,Debate
7,S008,61-4844,Mohamed Adel,EMS,Cairo,Music
8,S009,61-4844,Mohamed Adel,EMS,Cairo,Robotics
20,S024,55-2992,Mohamed Adel,IET,Alexandria,Robotics
30,S035,61-4844,Mohamed Adel,EMS,Cairo,Debate


## Save

In [17]:
df.to_csv("club_signups_clean.csv", index=False)
print(df.shape)
df

(32, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False


In [18]:
checks = pd.Series({
    "one row per student per club": not df.duplicated(["student_id", "club"]).any(),
    "faculty is canonical": df["faculty"].isin({"MET", "IET", "EMS", "Pharmacy", "Management"}).all(),
    "club is canonical": df["club"].isin({"Robotics", "Debate", "Music", "Football", "Chess"}).all(),
    "city is canonical": df["city"].isin({"Cairo", "Giza", "Alexandria"}).all(),
    "names have no extra spaces, Title Case": (df["full_name"] == df["full_name"].str.split().str.join(" ").str.title()).all(),
    "emails trimmed and lower-case": (df["email"] == df["email"].str.strip().str.lower()).all(),
    "fee_paid is boolean": df["fee_paid"].dtype == bool,
    "signed_up_at is datetime and complete": str(df["signed_up_at"].dtype).startswith("datetime64") and df["signed_up_at"].notna().all(),
    "39 rows → 32 rows": len(raw) == 39 and len(df) == 32,
})
assert checks.all(), checks[~checks]
checks

one row per student per club              True
faculty is canonical                      True
club is canonical                         True
city is canonical                         True
names have no extra spaces, Title Case    True
emails trimmed and lower-case             True
fee_paid is boolean                       True
signed_up_at is datetime and complete     True
39 rows → 32 rows                         True
dtype: bool